# StepManAI — Training

Trains the step **placement** and **selection** models. Fully self-contained:
the first run downloads ~19 official DDR simfile packs from Zenius-I-Vanisher,
builds the feature dataset on the Colab disk, and saves a reusable copy to your
Google Drive (`MyDrive/StepManAI/stepmanai_dataset.tar`) so later runs skip the
download. Checkpoints go to `MyDrive/StepManAI/checkpoints/` (used by the
generate notebook).

**Before running:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
#@title 1. Setup + dataset (downloads DDR packs from ZIV on first run)
!nvidia-smi -L
!pip install -q soundfile
from google.colab import drive
drive.mount('/content/drive')
import glob, os
if not os.path.exists('/content/StepManAI'):
    !git clone -q https://github.com/Mrman67/StepManAI.git /content/StepManAI

DRIVE_TAR = '/content/drive/MyDrive/StepManAI/stepmanai_dataset.tar'
if os.path.exists('/content/data/labels.pkl'):
    print('dataset already prepared in this session')
elif os.path.exists(DRIVE_TAR):
    print('loading prebuilt dataset from Drive...')
    !mkdir -p /content/data && tar -xf "{DRIVE_TAR}" -C /content/data
else:
    # ZIV categoryids for the 19 official DDR arcade mixes
    PACKS = {37:'1st', 32:'2nd', 38:'3rd', 39:'4th', 303:'4thplus', 30:'5th',
             40:'max', 31:'max2', 41:'extreme', 1:'supernova', 77:'supernova2',
             295:'x', 546:'x2', 802:'x3', 1148:'a', 1292:'a20', 1293:'a20plus',
             1509:'a3', 1709:'world'}
    os.makedirs('/content/Songs', exist_ok=True)
    for cid, name in PACKS.items():
        dst = f'/content/Songs/{name}'
        if os.path.exists(dst):
            continue
        print('downloading pack:', name, flush=True)
        url = f'https://zenius-i-vanisher.com/v5.2/download.php?type=ddrpack&categoryid={cid}'
        !wget -q -O /content/pack.zip "{url}" && mkdir -p "{dst}" && unzip -qo /content/pack.zip -d "{dst}" && rm /content/pack.zip
    print('parsing simfiles + extracting audio features (~30-40 min)...')
    os.environ['STEPMANAI_SONGS'] = '/content/Songs'
    os.environ['STEPMANAI_INDEX'] = '/content/data/index.json'
    os.environ['STEPMANAI_CACHE'] = '/content/data/cache_u8'
    os.environ['STEPMANAI_LABELS'] = '/content/data/labels.pkl'
    os.environ['STEPMANAI_U8'] = '1'
    !cd /content/StepManAI && python scan_library.py && python build_cache.py
    print('saving dataset tar to Drive for future runs...')
    !mkdir -p /content/drive/MyDrive/StepManAI
    !tar -cf "{DRIVE_TAR}" -C /content/data cache_u8 labels.pkl

print(len(glob.glob('/content/data/cache_u8/*.npy')), 'songs ready')

In [ ]:
#@title 2. Train placement model (when steps happen)
epochs = 40 #@param {type:"integer"}
batch_size = 64 #@param {type:"integer"}
import os
os.environ['STEPMANAI_ROOT'] = '/content/StepManAI'
os.environ['STEPMANAI_CACHE'] = '/content/data/cache_u8'
os.environ['STEPMANAI_LABELS'] = '/content/data/labels.pkl'
os.environ['EPOCHS'] = str(epochs)
os.environ['BATCH'] = str(batch_size)
!cd /content/StepManAI && python train_placement.py

In [ ]:
#@title 3. Train selection model (which arrows)
epochs = 30 #@param {type:"integer"}
import os
os.environ['STEPMANAI_ROOT'] = '/content/StepManAI'
os.environ['STEPMANAI_LABELS'] = '/content/data/labels.pkl'
os.environ['EPOCHS'] = str(epochs)
!cd /content/StepManAI && python train_selection.py

In [ ]:
#@title 4. Save checkpoints to Drive
!mkdir -p /content/drive/MyDrive/StepManAI/checkpoints
!cp /content/StepManAI/checkpoints/placement.pt /content/drive/MyDrive/StepManAI/checkpoints/
!cp /content/StepManAI/checkpoints/selection.pt /content/drive/MyDrive/StepManAI/checkpoints/
!ls -la /content/drive/MyDrive/StepManAI/checkpoints/
print('done — open the generate notebook!')